Model 5 Memory Augmented Autoencoder (MemAE)


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import sys

drive.mount('/content/drive')

DATA_ROOT = Path("/content/UCSD_Anomaly_Dataset/UCSD_Anomaly_Dataset")
PED1_PATH = DATA_ROOT / "UCSDped1"
PED2_PATH = DATA_ROOT / "UCSDped2"
ZIP_PATH = Path("/content/drive/MyDrive/SurveillanceAnomalyDetection/UCSD_Anomaly_Dataset.zip")
EXTRACT_PATH = Path("/content/UCSD_Anomaly_Dataset")

if not (PED1_PATH.exists() and PED2_PATH.exists()):
  with ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_PATH)

GH_TOKEN = userdata.get("GH_PAT")
REPO_DIR = Path("/content/drive/MyDrive/SurveillanceAnomalyDetection/repo")
REPO_URL = f"https://{GH_TOKEN}@github.com/Rishabh-G-Shetye/SurveillanceAnomalyDetection.git"
if not REPO_DIR.exists():
  !git clone {REPO_URL} "{REPO_DIR}"
%cd "{REPO_DIR}"

sys.path.insert(0, str(REPO_DIR / "src"))

df = pd.read_csv(REPO_DIR / "data" / "metadata.csv")
from data.ucsd_dataset import PreprocessConfig, UCSDClipDataset

ped1_config = PreprocessConfig(target_size=(128, 128), window_length=8, stride=4)
train_ds = UCSDClipDataset(df, DATA_ROOT, "Ped1", "Train", ped1_config)
test_ds = UCSDClipDataset(df, DATA_ROOT, "Ped1", "Test", ped1_config)
print("Train clips:", len(train_ds), "| Test clips:", len(test_ds))

import torch
from torch.utils.data import DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

## 1b. Ensure Persistency

In [ ]:
import os
os.makedirs(REPO_DIR / "src" / "utils", exist_ok=True)
(REPO_DIR / "src" / "utils" / "__init__.py").touch()

In [ ]:
%%writefile "{REPO_DIR}/src/utils/persistence.py"
"""
Model/result persistence -- local version (replaces Colab Drive paths).

Checkpoints are saved under PROJECT_ROOT/models/
Results and figures under PROJECT_ROOT/outputs/
"""
import json
from pathlib import Path
import numpy as np
import torch

# Project root is two levels up from this file (src/utils/persistence.py -> project root)
PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent

CKPT_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "outputs" / "logs"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"


def save_checkpoint(model: torch.nn.Module, name: str) -> Path:
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    path = CKPT_DIR / f"{name}.pt"
    torch.save(model.state_dict(), path)
    print(f"  Checkpoint saved: {path}")
    return path


def load_checkpoint(model: torch.nn.Module, name: str, device=None) -> torch.nn.Module:
    path = CKPT_DIR / f"{name}.pt"
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    return model.to(device)


def save_results(name: str, results: dict, scores: np.ndarray, labels: np.ndarray) -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(RESULTS_DIR / f"{name}_metrics.json", "w") as f:
        json.dump(results, f, indent=2, default=float)
    np.savez(RESULTS_DIR / f"{name}_scores.npz", scores=scores, labels=labels)
    print(f"  Results saved: {RESULTS_DIR / name}")


def load_results(name: str):
    with open(RESULTS_DIR / f"{name}_metrics.json") as f:
        results = json.load(f)
    npz = np.load(RESULTS_DIR / f"{name}_scores.npz")
    return results, npz["scores"], npz["labels"]


def try_load_results(name: str):
    """Like load_results, but returns None if the artifact doesn't exist."""
    try:
        return load_results(name)
    except FileNotFoundError:
        return None


def save_metrics_only(name: str, results: dict) -> Path:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    path = RESULTS_DIR / f"{name}_metrics.json"
    with open(path, "w") as f:
        json.dump(results, f, indent=2, default=float)
    return path


def save_figure(fig, name: str) -> Path:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURES_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  Figure saved: {path}")
    return path


## 2. Memory-Augmented Autoencoder (MemAE)

### Numerical Stability Fix:
- In Gong et al.'s original shrinkage formula, dividing by $|\text{attn} - \lambda| + 1e-12$ caused gradient explosions and `NaN` losses whenever attention weights hovered around the threshold.
- We replaced this with `torch.where(attn > thresh, attn, 0)` with a fallback to standard softmax if all slots are pruned. This resolved gradient stability and allowed smooth training (reaching **80.5% AUC on Ped2**).


In [ ]:
%%writefile "{REPO_DIR}/src/models/memory_ae.py"
"""
Memory-Augmented Autoencoder (MemAE) for Video Anomaly Detection (Gong et al., ICCV 2019).

Standard autoencoders sometimes generalize "too well" and reconstruct anomalies with low error.
MemAE addresses this by routing the latent representation through a memory matrix
containing learned prototype patterns of normal video. The latent representation is
reconstructed as a sparse linear combination of normal memory items. Because anomalous
patterns cannot be represented by the normal memory prototypes, they fail to reconstruct.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from .base import BaseAnomalyModel


class MemoryModule(nn.Module):
    """
    Learned memory prototype bank with hard-shrinkage addressing.
    Forces sparse attention weights so latent codes only use a few normal prototypes.
    """

    def __init__(self, num_slots: int = 100, slot_dim: int = 512, shrink_thresh: float = 0.0025):
        super().__init__()
        self.num_slots = num_slots
        self.slot_dim = slot_dim
        self.shrink_thresh = shrink_thresh
        # Memory matrix: (num_slots, slot_dim)
        self.memory = nn.Parameter(torch.empty(num_slots, slot_dim))
        nn.init.kaiming_uniform_(self.memory, a=math.sqrt(5))

    def forward(self, z: torch.Tensor):
        # Compute cosine similarity between latent vectors and memory items
        z_norm = F.normalize(z, dim=1)
        mem_norm = F.normalize(self.memory, dim=1)
        sim = z_norm @ mem_norm.t()  # shape: (N, num_slots)
        attn = F.softmax(sim, dim=1)

        # Stability fix: replace original paper's formula with stable torch.where
        # The original formula divided by |attn - thresh| + 1e-12, which caused gradient explosion
        # when attention values were near the threshold.
        attn = torch.where(attn > self.shrink_thresh, attn, torch.zeros_like(attn))
        sum_attn = attn.sum(dim=1, keepdim=True)
        # Fallback to standard softmax if all slots are pruned below threshold
        attn = torch.where(sum_attn > 1e-8, attn / (sum_attn + 1e-12), F.softmax(sim, dim=1))

        # Reconstruct latent code using memory prototypes
        z_hat = attn @ self.memory  # shape: (N, slot_dim)
        return z_hat, attn


class MemAE(BaseAnomalyModel):
    def __init__(
        self,
        in_channels: int = 1,
        latent_dim: int = 64,
        num_slots: int = 100,
        feat_size: int = 16,
        shrink_thresh: float = 0.0025,
        entropy_weight: float = 0.0002,
        score_agg: str = "mean",
    ):
        super().__init__()
        assert score_agg in ("mean", "max")
        self.score_agg = score_agg
        self.entropy_weight = entropy_weight
        self.latent_dim = latent_dim
        self.feat_size = feat_size
        self.shrink_thresh = shrink_thresh

        # 2D Encoder: downsample 128x128 -> 16x16
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(64, latent_dim, 4, 2, 1), nn.ReLU(),
        )
        self.memory = MemoryModule(
            num_slots=num_slots,
            slot_dim=latent_dim * feat_size * feat_size,
            shrink_thresh=shrink_thresh,
        )
        # 2D Decoder: upsample 16x16 -> 128x128
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, in_channels, 4, 2, 1), nn.Tanh(),
        )

    def forward_frame(self, x: torch.Tensor):
        B, C, H, W = x.shape
        z = self.encoder(x)
        z_flat = z.view(B, -1)
        z_hat_flat, attn = self.memory(z_flat)
        z_hat = z_hat_flat.view(B, self.latent_dim, self.feat_size, self.feat_size)
        recon = self.decoder(z_hat)
        return recon, attn

    def forward(self, clip: torch.Tensor) -> torch.Tensor:
        B, T, C, H, W = clip.shape
        x_flat = clip.view(B * T, C, H, W)
        recon, _ = self.forward_frame(x_flat)
        return recon.view(B, T, C, H, W)

    def compute_loss(self, clip: torch.Tensor) -> torch.Tensor:
        B, T, C, H, W = clip.shape
        x_flat = clip.view(B * T, C, H, W)
        recon, attn = self.forward_frame(x_flat)
        # Reconstruction MSE
        recon_loss = F.mse_loss(recon, x_flat)
        # Memory entropy loss: encourages sparse memory usage
        entropy_loss = (-attn * torch.log(attn + 1e-12)).sum(dim=-1).mean()
        return recon_loss + self.entropy_weight * entropy_loss

    def per_frame_anomaly_score(self, clip: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            recon = self.forward(clip)
            return ((recon - clip) ** 2).mean(dim=(2, 3, 4))


## 3. Train MemAE

In [ ]:
from models.memory_ae import MemAE
from training.trainer import train_model

model_new = MemAE()
history_new = train_model(model_new, train_loader, num_epochs=15, lr=1e-3)

## 4. Evaluate + Persist MemAE

In [ ]:
import torch
import numpy as np
from evaluation.metrics import evaluate_scores
from utils.persistence import save_checkpoint, save_results

device = "cuda" if torch.cuda.is_available() else "cpu"
model_new.eval()

new_scores, new_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        clip = batch["clip"].to(device)
        new_scores.append(model_new.anomaly_score(clip).cpu().numpy())
        new_labels.append(batch["clip_label"].numpy())

new_scores = np.concatenate(new_scores)
new_labels = np.concatenate(new_labels)

results_new = evaluate_scores(new_labels, new_scores)
print("MemAE results:", results_new)

save_checkpoint(model_new, "memae_baseline")
save_results("memae_baseline", results_new, new_scores, new_labels)

## 5. Full Dual-Dataset Benchmark (All 5 Models on UCSD Ped1 & Ped2)

### Complete Benchmark Comparison:
Here is the final quantitative comparison across all 5 architectures on both UCSD Ped1 and UCSD Ped2 (8 training epochs, NVIDIA T1200 GPU):

| Model | Architecture Type | Ped1 Frame AUC | Ped1 F1 | Ped2 Frame AUC | Ped2 F1 | Ped2 Event Recall | Latency (ms/frame) | Throughput (FPS) |
|---|---|:---:|:---:|:---:|:---:|:---:|:---:|:---:| 
| **FramePrediction** | Future Frame Prediction | **0.7604** | **0.7772** | **0.8422** | **0.9172** | **100%** | 0.70 ms | 1,426 FPS |
| **MemAE** | Memory-Augmented Bottleneck | **0.7402** | **0.7607** | **0.8054** | **0.9090** | **100%** | 0.40 ms | 2,498 FPS |
| **TransformerAE** | Spatiotemporal Attention | **0.7268** | **0.7483** | **0.7952** | **0.9098** | **100%** | **0.25 ms** | **3,982 FPS** |
| **ConvLSTM-AE** | Spatiotemporal Recurrent | **0.6953** | **0.7390** | **0.7606** | **0.9022** | **100%** | 0.76 ms | 1,318 FPS |
| **ConvAE** | Spatial Reconstruction Baseline | **0.6969** | **0.7428** | **0.7291** | **0.9000** | **100%** | 0.37 ms | 2,679 FPS |

### Final Engineering Observations:
1. **Future Frame Prediction achieved the highest accuracy** (0.7604 Ped1, 0.8422 Ped2) because learning motion dynamics is far more sensitive to unexpected velocities than pure spatial reconstruction.
2. **TransformerAE was the most computationally efficient** (3,982 FPS, 0.25 ms/frame), easily beating the 30 FPS real-time threshold.
3. **All models achieved 100% Event Recall on Ped2**, intercepting every single ground-truth anomaly event in the test set.


In [ ]:
from utils.persistence import try_load_results, save_figure
from evaluation.visualize import plot_roc_curves, plot_metric_comparison

# Every notebook trains on the same Ped1 Test split (same PreprocessConfig
# throughout the project), so persisted label arrays are interchangeable --
# this cell picks up whichever of these have been trained and persisted so
# far, and simply skips any that haven't (e.g. running notebook 06 before
# 07/08 exist yet is fine).
ALL_MODELS = {
    "conv_ae_baseline": "ConvAE",
    "convlstm_ae_baseline": "ConvLSTM-AE",
    "transformer_ae_baseline": "TransformerAE",
    "frame_prediction_baseline": "FramePrediction",
    "memae_baseline": "MemAE",
}

loaded = {}
for ckpt_name, display in ALL_MODELS.items():
    found = try_load_results(ckpt_name)
    if found is not None:
        loaded[display] = found  # (results, scores, labels)

print("Models found for comparison:", list(loaded.keys()))

any_labels = next(iter(loaded.values()))[2]
scores_by_model = {name: res[1] for name, res in loaded.items()}
results_by_model = {name: res[0] for name, res in loaded.items()}

fig_roc = plot_roc_curves(any_labels, scores_by_model)
save_figure(fig_roc, "roc_comparison_all_models")

fig_bar = plot_metric_comparison(results_by_model)
save_figure(fig_bar, "metric_comparison_all_models")

## 6. Score Timeline

In [ ]:
from models.conv_ae import ConvAE
from utils.persistence import load_checkpoint, save_figure
from evaluation.visualize import plot_score_timeline

conv_ae_ref = load_checkpoint(ConvAE(use_skip=False, score_agg="mean"), "conv_ae_baseline")

fig_t1 = plot_score_timeline(conv_ae_ref, test_ds, "Test003")   # reference ceiling
save_figure(fig_t1, "score_timeline_test003_convae_ref")

fig_t2 = plot_score_timeline(model_new, test_ds, "Test003")     # this notebook's model
save_figure(fig_t2, "score_timeline_test003_memae_baseline")

## 7. Frame-level, Event-level Accuracy and Inference Time

In [ ]:
from evaluation.metrics import frame_and_event_level_eval, measure_inference_time
from utils.persistence import save_metrics_only
import pandas as pd

sample_clip = test_ds[0]["clip"]  # (T, 1, H, W)

frame_event_ref = frame_and_event_level_eval(conv_ae_ref, test_ds, device=device)
frame_event_new = frame_and_event_level_eval(model_new, test_ds, device=device)

timing_ref = measure_inference_time(conv_ae_ref, sample_clip, device=device)
timing_new = measure_inference_time(model_new, sample_clip, device=device)

summary = {
    "ConvAE (reference)": {**frame_event_ref, **timing_ref},
    "MemAE": {**frame_event_new, **timing_new},
}

save_metrics_only("frame_event_efficiency_memae_baseline", summary)
pd.DataFrame(summary).T[["frame_auc_roc", "frame_f1", "event_precision_avg",
                          "event_recall_avg", "ms_per_frame"]]

In [ ]:
# %cd "{REPO_DIR}"
# !git add notebooks/08_ucsd_memory_ae.ipynb
# !git commit -m "feat: add MemAE model, train/evaluate/persist, compare against all prior models"
# !git push